In [1]:
# Cell 1: Environment Setup & Imports
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [2]:
# Cell 2: Sequence Generation Function
def create_sequences(df_ticker, seq_length):
    features = ['Close', 'Volume', 'Sentiment', 'SMA_20', 'RSI_14']
    target = 'Close'
    X, y = [], []
    for i in range(len(df_ticker) - seq_length):
        X.append(df_ticker[features].iloc[i : i + seq_length].values)
        y.append(df_ticker[target].iloc[i + seq_length])
    return np.array(X), np.array(y)


In [3]:
# Cell 3: BiLSTM (No Attention) Architecture
class StockPredictorBiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size=1, dropout_rate=0.2):
        super(StockPredictorBiLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout_rate
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(hidden_size * 2, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, output_size)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        # No attention: take only the final timestep representation
        last_timestep_out = lstm_out[:, -1, :] 
        out = self.dropout(last_timestep_out)
        return self.fc2(self.relu(self.fc1(out)))


In [4]:
# Cell 4: Automated Multi-Ticker & Stability Loop

# --- CONFIGS (Moved here to match standardized structure) ---
TICKERS = ['MSFT', 'DIS', 'WMT'] 
SEQ_LENGTHS = [5, 10, 20] # Triple testing
NUM_RUNS = 10
EPOCHS = 100
BATCH_SIZE = 128
LEARNING_RATE = 0.001

BASE_MODEL_DIR = "models/v2_no_attention"
BASE_RESULTS_DIR = "results/trainV2_no_attention"
os.makedirs(BASE_RESULTS_DIR, exist_ok=True)

for seq_len in SEQ_LENGTHS:
    print(f"\n" + "#"*60 + f"\nSTARTING V2 NO-ATTN EXPERIMENTS | SEQUENCE LENGTH: {seq_len}\n" + "#"*60)
    SEQ_RESULTS = []
    
    for ticker in TICKERS:
        print(f"\n>>> Processing: {ticker} (Seq: {seq_len})")
        
        file_path = f"datasets/{ticker}/{ticker}_DatasetV1.csv"
        ticker_df = pd.read_csv(file_path)
        ticker_df['Trading_Date'] = pd.to_datetime(ticker_df['Trading_Date'])
        
        # Split by Year Consistent with V1
        train_raw = ticker_df[ticker_df['Trading_Date'].dt.year <= 2022].copy()
        test_raw = ticker_df[ticker_df['Trading_Date'].dt.year == 2023].copy()
        
        # Scaling: Fit ONLY on training data
        scaler = MinMaxScaler()
        cols = ['Close', 'Volume', 'Sentiment', 'SMA_20', 'RSI_14']
        train_raw[cols] = scaler.fit_transform(train_raw[cols])
        test_raw[cols] = scaler.transform(test_raw[cols])
        
        # Prepare Data
        X_train, y_train = create_sequences(train_raw, seq_len)
        X_test, y_test = create_sequences(test_raw, seq_len)
        
        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
        X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

        for run in range(1, NUM_RUNS + 1):
            # BiLSTM Standard (No Attention)
            model = StockPredictorBiLSTM(input_size=5, hidden_size=128, num_layers=2).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
            criterion = nn.MSELoss()
            
            train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=128, shuffle=True)
            
            model.train()
            for epoch in range(EPOCHS):
                for xb, yb in train_loader:
                    optimizer.zero_grad()
                    criterion(model(xb), yb).backward()
                    optimizer.step()
            
            model.eval()
            with torch.no_grad():
                preds = model(X_test_t).cpu().numpy().flatten()
            
            r2 = r2_score(y_test, preds)
            mae = mean_absolute_error(y_test, preds)
            
            # Store Metrics
            SEQ_RESULTS.append({'Ticker': ticker, 'Run': run, 'Seq': seq_len, 'R2': r2, 'MAE': mae})
            
            # Save weights (.pth)
            save_path = os.path.join(BASE_MODEL_DIR, ticker)
            os.makedirs(save_path, exist_ok=True)
            torch.save(model.state_dict(), f"{save_path}/v2_no_att_run_{run}_seq{seq_len}.pth")
            
            print(f"  Run {run:02d}: R2={r2:.4f}")

    # Save CSV Results
    results_df = pd.DataFrame(SEQ_RESULTS)
    csv_path = os.path.join(BASE_RESULTS_DIR, f"v2_no_att_stability_results_{seq_len}.csv")
    results_df.to_csv(csv_path, index=False)
    
    print(f"\n[DONE] Seq {seq_len} results saved to: {csv_path}")
    display(results_df.groupby('Ticker')[['R2', 'MAE']].agg(['mean', 'std']))



############################################################
STARTING V2 NO-ATTN EXPERIMENTS | SEQUENCE LENGTH: 5
############################################################

>>> Processing: MSFT (Seq: 5)
  Run 01: R2=0.9788
  Run 02: R2=0.9615
  Run 03: R2=0.9797
  Run 04: R2=0.9589
  Run 05: R2=0.9714
  Run 06: R2=0.9798
  Run 07: R2=0.9493
  Run 08: R2=0.9825
  Run 09: R2=0.9600
  Run 10: R2=0.9605

>>> Processing: DIS (Seq: 5)
  Run 01: R2=0.8752
  Run 02: R2=0.9361
  Run 03: R2=0.8845
  Run 04: R2=0.7961
  Run 05: R2=0.9141
  Run 06: R2=0.8824
  Run 07: R2=0.9438
  Run 08: R2=0.9276
  Run 09: R2=0.9196
  Run 10: R2=0.9438

>>> Processing: WMT (Seq: 5)
  Run 01: R2=0.8719
  Run 02: R2=0.8989
  Run 03: R2=0.9164
  Run 04: R2=0.9387
  Run 05: R2=0.9534
  Run 06: R2=0.9322
  Run 07: R2=0.9125
  Run 08: R2=0.8269
  Run 09: R2=0.9300
  Run 10: R2=0.9257

[DONE] Seq 5 results saved to: results/trainV2_no_attention\v2_no_att_stability_results_5.csv


R2                 MAE          
            mean       std      mean       std
Ticker                                        
DIS     0.902333  0.045192  0.016270  0.004001
MSFT    0.968241  0.011597  0.022810  0.004973
WMT     0.910644  0.037078  0.027937  0.007076


############################################################
STARTING V2 NO-ATTN EXPERIMENTS | SEQUENCE LENGTH: 10
############################################################

>>> Processing: MSFT (Seq: 10)
  Run 01: R2=0.9722
  Run 02: R2=0.9746
  Run 03: R2=0.9773
  Run 04: R2=0.9821
  Run 05: R2=0.9738
  Run 06: R2=0.9778
  Run 07: R2=0.9774
  Run 08: R2=0.9802
  Run 09: R2=0.9659
  Run 10: R2=0.9535

>>> Processing: DIS (Seq: 10)
  Run 01: R2=0.8824
  Run 02: R2=0.8305
  Run 03: R2=0.9077
  Run 04: R2=0.9134
  Run 05: R2=0.8163
  Run 06: R2=0.8207
  Run 07: R2=0.9030
  Run 08: R2=0.9119
  Run 09: R2=0.7477
  Run 10: R2=0.8610

>>> Processing: WMT (Seq: 10)
  Run 01: R2=0.6726
  Run 02: R2=0.8741
  Run 03: R2=0.9455
  Run 04: R2=0.9418
  Run 05: R2=0.9050
  Run 06: R2=0.8864
  Run 07: R2=0.9334
  Run 08: R2=0.9480
  Run 09: R2=0.8888
  Run 10: R2=0.9105

[DONE] Seq 10 results saved to: results/trainV2_no_attention\v2_no_att_stability_results_10.csv


R2                 MAE          
            mean       std      mean       std
Ticker                                        
DIS     0.859453  0.054887  0.020678  0.004707
MSFT    0.973486  0.008356  0.019753  0.003464
WMT     0.890626  0.081100  0.030453  0.012407


############################################################
STARTING V2 NO-ATTN EXPERIMENTS | SEQUENCE LENGTH: 20
############################################################

>>> Processing: MSFT (Seq: 20)
  Run 01: R2=0.9759
  Run 02: R2=0.9775
  Run 03: R2=0.9742
  Run 04: R2=0.9699
  Run 05: R2=0.9725
  Run 06: R2=0.9668
  Run 07: R2=0.9405
  Run 08: R2=0.9757
  Run 09: R2=0.9549
  Run 10: R2=0.9787

>>> Processing: DIS (Seq: 20)
  Run 01: R2=0.8493
  Run 02: R2=0.9284
  Run 03: R2=0.8964
  Run 04: R2=0.9082
  Run 05: R2=0.8058
  Run 06: R2=0.8331
  Run 07: R2=0.8299
  Run 08: R2=0.8436
  Run 09: R2=0.8722
  Run 10: R2=0.8333

>>> Processing: WMT (Seq: 20)
  Run 01: R2=0.8561
  Run 02: R2=0.8846
  Run 03: R2=0.9486
  Run 04: R2=0.9004
  Run 05: R2=0.9241
  Run 06: R2=0.8919
  Run 07: R2=0.8847
  Run 08: R2=0.9276
  Run 09: R2=0.9405
  Run 10: R2=0.8516

[DONE] Seq 20 results saved to: results/trainV2_no_attention\v2_no_att_stability_results_20.csv


R2                 MAE          
            mean       std      mean       std
Ticker                                        
DIS     0.860008  0.039638  0.019414  0.003360
MSFT    0.968657  0.012082  0.019889  0.004372
WMT     0.901003  0.033559  0.028313  0.006362